## Converting bracken output to normalized ASV table
- this script is from nikea https://github.com/nikeaulrich/DR_SCTLD/blob/main/bracken_abundances_to_ASV.ipynb

### Species

#### Upload

In [15]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [10]:
os.chdir("/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken")

In [11]:
SAMPLES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_samples.txt"
samplelist=[]

# make samplelist
with open(SAMPLES) as samples:
    for line in samples:
        samplelist.append(line.strip())

# remove ''
sample_names_string = ','.join(samplelist)
print(len(samplelist))

222


In [16]:
# not sure yet what this is for 
BRACKENS=Path("/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken")

In [ ]:
# combine bracken outputs
# using nikea's downloaded bracken script - run in terminal

python /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/scripts/combine_bracken_outputs.py \
    --files $BRACKENS/*.bracken \
    --output $BRACKENS/all_merged_bracken_S.txt

In [14]:
BRACKENS

'/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken'

In [17]:
brackenfile="all_merged_bracken_S.txt"
otu_table=pd.read_csv(BRACKENS/brackenfile, sep='\t')


In [19]:
otu_table.set_index("taxonomy_id",inplace=True)

In [22]:
otu_table.head()

,name,taxonomy_lvl,052022_BEL_CBC_T1_10_PSTR.kreport2_S.bracken_num,052022_BEL_CBC_T1_10_PSTR.kreport2_S.bracken_frac,052022_BEL_CBC_T1_11_PSTR.kreport2_S.bracken_num,052022_BEL_CBC_T1_11_PSTR.kreport2_S.bracken_frac,052022_BEL_CBC_T1_12_MCAV.kreport2_S.bracken_num,052022_BEL_CBC_T1_12_MCAV.kreport2_S.bracken_frac,052022_BEL_CBC_T1_13_MCAV.kreport2_S.bracken_num,052022_BEL_CBC_T1_13_MCAV.kreport2_S.bracken_frac,...,122022_BEL_CBC_T4_8_PSTR.kreport2_S.bracken_num,122022_BEL_CBC_T4_8_PSTR.kreport2_S.bracken_frac,122022_BEL_CBC_T4_9_OFAV.kreport2_S.bracken_num,122022_BEL_CBC_T4_9_OFAV.kreport2_S.bracken_frac,7_11_Neg.kreport2_S.bracken_num,7_11_Neg.kreport2_S.bracken_frac,7_3_Neg.kreport2_S.bracken_num,7_3_Neg.kreport2_S.bracken_frac,Negative_extract_11-2-24.kreport2_S.bracken_num,Negative_extract_11-2-24.kreport2_S.bracken_frac
taxonomy_id,,,,,,,,,,,,,,,,,,,,,
3021011,Vibrio sp. SCSIO 43137,S,13256,0.01496,37,0.00007,1854,0.00222,0,0.00000,...,26,0.00002,196,0.00011,0,0.0,0,0.0,0,0.0
2819098,Vibrio sp. SCSIO 43153,S,2032,0.00229,123,0.00022,31,0.00004,0,0.00000,...,0,0.00000,47,0.00003,29,0.0,0,0.0,17,0.0
2819101,Vibrio sp. SCSIO 43136,S,260,0.00029,121,0.00022,58,0.00007,0,0.00000,...,149,0.00010,419,0.00024,0,0.0,0,0.0,0,0.0
2912314,Vibrio sp. JC009,S,99,0.00011,23,0.00004,78,0.00009,0,0.00000,...,17,0.00001,70,0.00004,0,0.0,0,0.0,0,0.0
2751178,Vibrio sp. B1FLJ16,S,91,0.00010,59,0.00011,28,0.00003,14,0.00006,...,76,0.00005,142,0.00008,0,0.0,0,0.0,0,0.0


Columns: \
Sample num: Number of reads estimated for that species  \
Sample frac: Fraction of total reads in the sample estimated for this species \
[NOTE: Fractions are of total reads classified, unclassified reads not accounted for]

In [23]:
# Make taxa otu id table 
otu_id=otu_table.iloc[:, 0:2]
otu_id
# export otu id 
otu_id.to_csv("otu_id_species.csv")
#output to /bracken

In [24]:
# drop name, taxa, and frac cols
drop_columns = [col for col in otu_table.columns if 'frac' in col]
drop_columns.extend(['name', 'taxonomy_lvl'])
otu_table_species=otu_table.drop(columns=drop_columns)

# rename to drop _num
#otu_table_species.columns.str.replace('_num', '')

# export otu table to csv
otu_table_species.to_csv("otu_table_species.csv")

#### Normalize 

I think I want to normalize based on number of reads classified for each sample (after filtering out human)

I previously normalized from total reads per sample, but then that includes all the unclassified reads as well as the human classified reads. As long as we are normalizing for differences in reads per sample I think this is better.

In [40]:
import re

In [31]:
os.getcwd()

'/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken'

In [ ]:
# slurm outputs are individual per sample
# combined in terminal using bash
cat /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/taxonomy/kraken/slurm*.out > slurm_combined.log

slurmdir='/work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/taxonomy/kraken/slurm_combined.log'
with open(slurmdir, 'r') as file:
    report_text = file.read()

In [44]:
# just to look at pattern of one of these outputs
file1 = "/work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/taxonomy/kraken/slurm-bracken-61280182_1.out"
with open(file1, 'r') as file:
    sample_text = file.read()
sample_text
# pattern is: "Total reads in sample: 27922584"

'Removing conda version miniforge3-24.7.1\nNo conda package cache directories found outside your home directory. To\nprevent conda from filling up your home directory, you can create a new\ndirectory at `/work/pi_<your_pi_name>/$USER/.conda/pkgs` and reload the module. \nNo conda environment directories found outside your home directory. To prevent\nconda from filling up your home directory, you can create a new directory at\n`/work/pi_<your_pi_name>/$USER/.conda/envs` and reload the module. \nLoading conda version miniforge3-24.7.1\n >> Checking for Valid Options...\n >> Running Bracken \n      >> python src/est_abundance.py -i /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/052022_BEL_CBC_T1_10_PSTR.kreport2 -o /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/052022_BEL_CBC_T1_10_PSTR.kreport2_S.bracken -k /datasets/bio/kraken2/PlusPF/database150mers.kmer_distrib -l S -t 10\n>> Checking report file: 

In [45]:
# Extract total number of reads using regular expression
    # try one single sample first
total_reads_pattern = re.compile(r'Total reads in sample: (\d+)')
sample_reads_matches = total_reads_pattern.findall(sample_text)
sample_reads_matches

['27922584']

In [48]:
# now try on all 
total_reads_matches = total_reads_pattern.findall(report_text)
total_reads_matches[:10]

['27922584',
 '8751821',
 '9074166',
 '8132481',
 '9745541',
 '8646246',
 '16121676',
 '2911230',
 '1861295',
 '21287461']

In [ ]:
### Jun 30, 2026 11:30pm this is where I stopped. ### 
### below is nikea's script ###

In [15]:
# Create a dictionary to map sample ID to total reads
total_reads_dict = {samplelist[i]: int(total_reads_matches[i]) for i in range(len(samplelist))}

# Display the updated DataFrame
print(total_reads_dict)

{'072023_Carolina_2023_Baya_053_MCAV_S3': 119041, '072023_Carolina_2023_Baya_055_MCAV_S4': 275961, '122021_Coralina_2021_Baya_003_MCAV_S27': 31441, '122021_Coralina_2021_Baya_005_MCAV_S28': 46677, '122021_Coralina_2021_Baya_007_MCAV_S29': 115215, '122021_Coralina_2021_Baya_017_MCAV_S30': 31184, '122021_Coralina_2021_Baya_023_MCAV_S31': 75058, '072023_Carolina_2023_Baya_052_DCYL_S9': 1152824, '072023_Carolina_2023_Baya_065_DCYL_S1': 68316, '072023_Carolina_2023_Baya_066_DCYL_S2': 157717, '122021_Coralina_2021_Baya_001_DCYL_S24': 43321, '122021_Coralina_2021_Baya_009_DCYL_S25': 25337, '122021_Coralina_2021_Baya_010_DCYL_S26': 20030, '122021_Coralina_2021_Baya_025_MMEA_S22': 53475, '122021_Coralina_2021_Baya_029_MMEA_S23': 21357, '122021_Coralina_2021_Baya_031_MMEA_S45': 62249, '072023_Carolina_2023_Baya_057_SSID_S7': 106217, '072023_Carolina_2023_Baya_059_SSID_S8': 163770, '072023_Carolina_2023_Baya_061_SSID_S5': 367338, '072023_Carolina_2023_Baya_063_SSID_S6': 804040, '122021_Coralina_2

In [17]:
# Convert main df to raw abundances 
normalized_otus=otu_table.loc[:,otu_table.columns.str.contains("num")]
#headers in the otu table need to match the all_sampleids.txt
normalized_otus.columns = normalized_otus.columns.str.replace('_S_filt.bracken_num', '')
normalized_otus

,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,072023_Carolina_2023_Baya_057_SSID_S7,072023_Carolina_2023_Baya_059_SSID_S8,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,122021_Coralina_2021_Baya_001_DCYL_S24,...,122021_Coralina_2021_Baya_019_SSID1_S46,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_023_MCAV_S31,122021_Coralina_2021_Baya_025_MMEA_S22,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_029_MMEA_S23,122021_Coralina_2021_Baya_031_MMEA_S45,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33
taxonomy_id,,,,,,,,,,,,,,,,,,,,,
588596,19466,1142,1539,716,1058,1321,910,747,792,262,...,289,3792,864,518,3845,345,664,1484,1219,849
311410,8042,0,23,0,10,53,201,0,17,15,...,19,16,0,0,28,0,0,25,10,16
5722,6083,575,731,316,538,750,595,270,312,123,...,3160,1854,362,224,1135,124,182,606,548,341
562,5839,742,1898,951,1954,2808,5742,311,696,878,...,552,1697,796,730,3338,303,1066,690,585,453
106654,5750,0,0,0,14,36,65,0,45,0,...,0,108,24,33,129,0,33,48,47,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3060540,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
1340801,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
2749077,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,53


In [18]:
# Convert the dictionary to a DataFrame and transpose it
sample_counts_df = pd.DataFrame(total_reads_dict, index=['Total_reads'])
sample_counts_df

,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,122021_Coralina_2021_Baya_003_MCAV_S27,122021_Coralina_2021_Baya_005_MCAV_S28,122021_Coralina_2021_Baya_007_MCAV_S29,122021_Coralina_2021_Baya_017_MCAV_S30,122021_Coralina_2021_Baya_023_MCAV_S31,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,...,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,122021_Coralina_2021_Baya_019_SSID1_S46,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33,122021_Coralina_2021_Baya_013_DLAB_S43,122021_Coralina_2021_Baya_015_DLAB_S44
Total_reads,119041,275961,31441,46677,115215,31184,75058,1152824,68316,157717,...,367338,804040,259683,385846,411063,128977,122740,83531,64881,523898


In [19]:
# Concatenate the original DataFrame and the new DataFrame with the total reads row
normalized_otus = pd.concat([normalized_otus, sample_counts_df],axis=0)
normalized_otus

,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,072023_Carolina_2023_Baya_057_SSID_S7,072023_Carolina_2023_Baya_059_SSID_S8,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,122021_Coralina_2021_Baya_001_DCYL_S24,...,122021_Coralina_2021_Baya_019_SSID1_S46,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_023_MCAV_S31,122021_Coralina_2021_Baya_025_MMEA_S22,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_029_MMEA_S23,122021_Coralina_2021_Baya_031_MMEA_S45,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33
588596,19466,1142,1539,716,1058,1321,910,747,792,262,...,289,3792,864,518,3845,345,664,1484,1219,849
311410,8042,0,23,0,10,53,201,0,17,15,...,19,16,0,0,28,0,0,25,10,16
5722,6083,575,731,316,538,750,595,270,312,123,...,3160,1854,362,224,1135,124,182,606,548,341
562,5839,742,1898,951,1954,2808,5742,311,696,878,...,552,1697,796,730,3338,303,1066,690,585,453
106654,5750,0,0,0,14,36,65,0,45,0,...,0,108,24,33,129,0,33,48,47,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1340801,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
2749077,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,53
3378073,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,35
2170403,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10


In [20]:
# Normalize: divide reads by total reads and multiply by 1 million
normalized_otu = (normalized_otus / normalized_otus.loc['Total_reads'] * 1E6)
# replace total reads row again 
normalized_otu.loc['Total_reads',:]=normalized_otus.loc['Total_reads',:]
normalized_otu["Sum"]=normalized_otu.sum(axis=1)
normalized_otu.sort_values(by="Sum", ascending=False)

,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,072023_Carolina_2023_Baya_057_SSID_S7,072023_Carolina_2023_Baya_059_SSID_S8,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,122021_Coralina_2021_Baya_001_DCYL_S24,...,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_023_MCAV_S31,122021_Coralina_2021_Baya_025_MMEA_S22,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_029_MMEA_S23,122021_Coralina_2021_Baya_031_MMEA_S45,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33,Sum
Total_reads,1.152824e+06,119041.000000,275961.000000,106217.000000,163770.000000,367338.000000,804040.000000,68316.000000,157717.000000,43321.000000,...,385846.000000,75058.000000,53475.000000,411063.000000,21357.000000,62249.000000,128977.000000,122740.000000,83531.000000,5.721187e+06
2057741,1.127666e+03,319.217749,148.571718,329.514108,305.306222,313.063173,532.311825,307.395047,0.000000,17705.039127,...,2016.348491,6941.298729,24385.226741,1471.793861,38675.843986,66764.124725,4442.652566,22942.805931,9828.686356,5.144996e+05
5722,5.276608e+03,4830.268563,2648.925029,2975.041660,3285.094950,2041.716348,740.012935,3952.222027,1978.226824,2839.269638,...,4805.025839,4822.936929,4188.873305,2761.133938,5806.058903,2923.741747,4698.512138,4464.722177,4082.316745,3.026703e+05
562,5.064954e+03,6233.146563,6877.783455,8953.369046,11931.367161,7644.186009,7141.435749,4552.374261,4412.967530,20267.306849,...,4398.127750,10605.132031,13651.238897,8120.409767,14187.385869,17124.773089,5349.791048,4766.172397,5423.136321,2.629771e+05
588596,1.688549e+04,9593.333389,5576.874993,6740.917179,6460.279660,3596.143062,1131.784488,10934.480942,5021.652707,6047.875164,...,9827.755115,11511.098084,9686.769518,9353.797350,16153.954207,10666.838021,11505.927413,9931.562653,10163.891250,2.586352e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3127115,8.674351e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.674351e+00
2560434,8.674351e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.674351e+00
2968831,8.674351e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.674351e+00
2051,8.674351e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.674351e+00


In [21]:
# export otu table to csv
normalized_otu.to_csv("otu_normtable_species.csv")

This otu table still has Total_reads row and Sum column. These should be removed and add taxonomy_id label to the first column

### Now let's do Kingdom level

starting at step 7: reading in the kingdom otu table this time

In [22]:
otu_table=pd.read_csv("/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/bracken_ready/all_merged_bracken_K.txt", sep='\t')

In [23]:
otu_table.set_index("taxonomy_id",inplace=True)

In [24]:
otu_table.head()

,name,taxonomy_lvl,072023_Carolina_2023_Baya_052_DCYL_S9_K_filt.bracken_num,072023_Carolina_2023_Baya_052_DCYL_S9_K_filt.bracken_frac,072023_Carolina_2023_Baya_053_MCAV_S3_K_filt.bracken_num,072023_Carolina_2023_Baya_053_MCAV_S3_K_filt.bracken_frac,072023_Carolina_2023_Baya_055_MCAV_S4_K_filt.bracken_num,072023_Carolina_2023_Baya_055_MCAV_S4_K_filt.bracken_frac,072023_Carolina_2023_Baya_057_SSID_S7_K_filt.bracken_num,072023_Carolina_2023_Baya_057_SSID_S7_K_filt.bracken_frac,...,122021_Coralina_2021_Baya_029_MMEA_S23_K_filt.bracken_num,122021_Coralina_2021_Baya_029_MMEA_S23_K_filt.bracken_frac,122021_Coralina_2021_Baya_031_MMEA_S45_K_filt.bracken_num,122021_Coralina_2021_Baya_031_MMEA_S45_K_filt.bracken_frac,122021_Coralina_2021_Baya_033_PSTR7_S35_K_filt.bracken_num,122021_Coralina_2021_Baya_033_PSTR7_S35_K_filt.bracken_frac,122021_Coralina_2021_Baya_035_PSTR_S32_K_filt.bracken_num,122021_Coralina_2021_Baya_035_PSTR_S32_K_filt.bracken_frac,122021_Coralina_2021_Baya_037_PSTR_S33_K_filt.bracken_num,122021_Coralina_2021_Baya_037_PSTR_S33_K_filt.bracken_frac
taxonomy_id,,,,,,,,,,,,,,,,,,,,,
3379134,Pseudomonadati,K,580304,0.49562,70881,0.50194,198189,0.65373,76820,0.56019,...,23646,0.57002,51278,0.59860,77956,0.50126,79320,0.53365,58084,0.52863
1783272,Bacillati,K,401741,0.34312,47538,0.33664,75913,0.25040,45852,0.33437,...,12697,0.30608,25016,0.29203,54680,0.35159,48854,0.32868,36972,0.33649
4751,Fungi,K,134770,0.11510,16585,0.11745,20395,0.06727,9442,0.06885,...,3592,0.08659,6248,0.07294,16132,0.10373,14524,0.09771,10482,0.09540
3366610,Methanobacteriati,K,17229,0.01471,1943,0.01376,2704,0.00892,1876,0.01368,...,457,0.01102,890,0.01039,2352,0.01512,2080,0.01399,1604,0.01460
3384189,Fusobacteriati,K,12800,0.01093,1262,0.00894,1828,0.00603,740,0.00540,...,336,0.00810,770,0.00899,1330,0.00855,1208,0.00813,834,0.00759


In [25]:
# Make taxa otu id table 
otu_id=otu_table.iloc[:, 0:2]
otu_id
# export otu id 
otu_id.to_csv("otu_id_species_kingdom.csv")
#output to /bracken_ready

In [26]:
# drop name, taxa, and frac cols
drop_columns = [col for col in otu_table.columns if 'frac' in col]
drop_columns.extend(['name', 'taxonomy_lvl'])
otu_table_species=otu_table.drop(columns=drop_columns)

# rename to drop _num
#otu_table_species.columns.str.replace('_num', '')

# export otu table to csv
otu_table_species.to_csv("otu_table_kingdom.csv")

In [27]:
# add column for number of classified reads per sample (doing species level right now)

import re
slurmdir='/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-bracken-filtering-K-60569484.out'
#make sure the order in which kraken.outs are merged are in the same order as the samplelist

In [28]:
# Read the output report and extract the total number of reads
with open(slurmdir, 'r') as file:
    report_text = file.read()

In [43]:
report_text

'Removing conda version miniforge3-24.7.1\nNo conda package cache directories found outside your home directory. To\nprevent conda from filling up your home directory, you can create a new\ndirectory at `/work/pi_<your_pi_name>/$USER/.conda/pkgs` and reload the module. \nNo conda environment directories found outside your home directory. To prevent\nconda from filling up your home directory, you can create a new directory at\n`/work/pi_<your_pi_name>/$USER/.conda/envs` and reload the module. \nLoading conda version miniforge3-24.7.1\n >> Checking for Valid Options...\n >> Running Bracken \n      >> python src/est_abundance.py -i /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/052022_BEL_CBC_T1_10_PSTR.kreport2 -o /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/052022_BEL_CBC_T1_10_PSTR.kreport2_S.bracken -k /datasets/bio/kraken2/PlusPF/database150mers.kmer_distrib -l S -t 10\n>> Checking report file: 

In [29]:
# Extract total number of reads using regular expression
total_reads_pattern = re.compile(r'(\d+) reads remaining')
total_reads_matches = total_reads_pattern.findall(report_text)
total_reads_matches

['141213',
 '303168',
 '49947',
 '73825',
 '139336',
 '52193',
 '94263',
 '1170856',
 '88588',
 '187011',
 '74574',
 '43338',
 '36381',
 '74346',
 '41483',
 '85663',
 '137131',
 '192320',
 '394542',
 '827632',
 '294164',
 '409942',
 '440200',
 '155520',
 '148638',
 '109877',
 '91820',
 '551823']

In [30]:
# Create a dictionary to map sample ID to total reads
total_reads_dict = {samplelist[i]: int(total_reads_matches[i]) for i in range(len(samplelist))}

# Display the updated DataFrame
print(total_reads_dict)

{'072023_Carolina_2023_Baya_053_MCAV_S3': 141213, '072023_Carolina_2023_Baya_055_MCAV_S4': 303168, '122021_Coralina_2021_Baya_003_MCAV_S27': 49947, '122021_Coralina_2021_Baya_005_MCAV_S28': 73825, '122021_Coralina_2021_Baya_007_MCAV_S29': 139336, '122021_Coralina_2021_Baya_017_MCAV_S30': 52193, '122021_Coralina_2021_Baya_023_MCAV_S31': 94263, '072023_Carolina_2023_Baya_052_DCYL_S9': 1170856, '072023_Carolina_2023_Baya_065_DCYL_S1': 88588, '072023_Carolina_2023_Baya_066_DCYL_S2': 187011, '122021_Coralina_2021_Baya_001_DCYL_S24': 74574, '122021_Coralina_2021_Baya_009_DCYL_S25': 43338, '122021_Coralina_2021_Baya_010_DCYL_S26': 36381, '122021_Coralina_2021_Baya_025_MMEA_S22': 74346, '122021_Coralina_2021_Baya_029_MMEA_S23': 41483, '122021_Coralina_2021_Baya_031_MMEA_S45': 85663, '072023_Carolina_2023_Baya_057_SSID_S7': 137131, '072023_Carolina_2023_Baya_059_SSID_S8': 192320, '072023_Carolina_2023_Baya_061_SSID_S5': 394542, '072023_Carolina_2023_Baya_063_SSID_S6': 827632, '122021_Coralina_2

In [31]:
# Convert main df to raw abundances 
normalized_otus=otu_table.loc[:,otu_table.columns.str.contains("num")]
#headers in the otu table need to match the all_sampleids.txt
normalized_otus.columns = normalized_otus.columns.str.replace('_K_filt.bracken_num', '')
normalized_otus

,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,072023_Carolina_2023_Baya_057_SSID_S7,072023_Carolina_2023_Baya_059_SSID_S8,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,122021_Coralina_2021_Baya_001_DCYL_S24,...,122021_Coralina_2021_Baya_019_SSID1_S46,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_023_MCAV_S31,122021_Coralina_2021_Baya_025_MMEA_S22,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_029_MMEA_S23,122021_Coralina_2021_Baya_031_MMEA_S45,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33
taxonomy_id,,,,,,,,,,,,,,,,,,,,,
3379134,580304,70881,198189,76820,104567,228686,512160,47851,116318,43897,...,181218,203924,45794,41490,220647,23646,51278,77956,79320,58084
1783272,401741,47538,75913,45852,65581,139454,287488,27315,55150,24735,...,94221,145265,33608,23570,156360,12697,25016,54680,48854,36972
4751,134770,16585,20395,9442,14973,17422,15064,9082,10555,3878,...,12864,42506,10128,6092,44295,3592,6248,16132,14524,10482
3366610,17229,1943,2704,1876,2768,3856,6681,1069,1587,724,...,2051,6118,1280,868,7738,457,890,2352,2080,1604
3384189,12800,1262,1828,740,1159,1286,1523,733,829,301,...,594,4159,1588,634,2768,336,770,1330,1208,834
2731360,6339,536,751,541,823,933,1049,536,507,224,...,896,1745,378,334,1837,152,208,835,554,422
3384194,5714,735,1080,492,750,1239,2262,519,715,306,...,1745,1769,391,264,1958,175,301,670,593,431
1783275,5290,788,1096,904,951,909,850,849,809,300,...,181,2491,572,704,2411,244,672,943,925,650
2732005,3104,507,656,238,386,476,335,543,453,145,...,155,962,277,322,1586,124,179,355,347,212


In [32]:
# Convert the dictionary to a DataFrame and transpose it
sample_counts_df = pd.DataFrame(total_reads_dict, index=['Total_reads'])
sample_counts_df

,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,122021_Coralina_2021_Baya_003_MCAV_S27,122021_Coralina_2021_Baya_005_MCAV_S28,122021_Coralina_2021_Baya_007_MCAV_S29,122021_Coralina_2021_Baya_017_MCAV_S30,122021_Coralina_2021_Baya_023_MCAV_S31,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,...,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,122021_Coralina_2021_Baya_019_SSID1_S46,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33,122021_Coralina_2021_Baya_013_DLAB_S43,122021_Coralina_2021_Baya_015_DLAB_S44
Total_reads,141213,303168,49947,73825,139336,52193,94263,1170856,88588,187011,...,394542,827632,294164,409942,440200,155520,148638,109877,91820,551823


In [33]:
# Concatenate the original DataFrame and the new DataFrame with the total reads row
normalized_otus = pd.concat([normalized_otus, sample_counts_df],axis=0)
normalized_otus

,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,072023_Carolina_2023_Baya_057_SSID_S7,072023_Carolina_2023_Baya_059_SSID_S8,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,122021_Coralina_2021_Baya_001_DCYL_S24,...,122021_Coralina_2021_Baya_019_SSID1_S46,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_023_MCAV_S31,122021_Coralina_2021_Baya_025_MMEA_S22,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_029_MMEA_S23,122021_Coralina_2021_Baya_031_MMEA_S45,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33
3379134,580304,70881,198189,76820,104567,228686,512160,47851,116318,43897,...,181218,203924,45794,41490,220647,23646,51278,77956,79320,58084
1783272,401741,47538,75913,45852,65581,139454,287488,27315,55150,24735,...,94221,145265,33608,23570,156360,12697,25016,54680,48854,36972
4751,134770,16585,20395,9442,14973,17422,15064,9082,10555,3878,...,12864,42506,10128,6092,44295,3592,6248,16132,14524,10482
3366610,17229,1943,2704,1876,2768,3856,6681,1069,1587,724,...,2051,6118,1280,868,7738,457,890,2352,2080,1604
3384189,12800,1262,1828,740,1159,1286,1523,733,829,301,...,594,4159,1588,634,2768,336,770,1330,1208,834
2731360,6339,536,751,541,823,933,1049,536,507,224,...,896,1745,378,334,1837,152,208,835,554,422
3384194,5714,735,1080,492,750,1239,2262,519,715,306,...,1745,1769,391,264,1958,175,301,670,593,431
1783275,5290,788,1096,904,951,909,850,849,809,300,...,181,2491,572,704,2411,244,672,943,925,650
2732005,3104,507,656,238,386,476,335,543,453,145,...,155,962,277,322,1586,124,179,355,347,212
2732397,1350,98,156,0,0,0,0,0,0,0,...,0,291,110,0,28,0,0,0,0,0


In [34]:
# Normalize: divide reads by total reads and multiply by 1 million
normalized_otu = (normalized_otus / normalized_otus.loc['Total_reads'] * 1E6)
# replace total reads row again 
normalized_otu.loc['Total_reads',:]=normalized_otus.loc['Total_reads',:]
normalized_otu["Sum"]=normalized_otu.sum(axis=1)
normalized_otu.sort_values(by="Sum", ascending=False)

,072023_Carolina_2023_Baya_052_DCYL_S9,072023_Carolina_2023_Baya_053_MCAV_S3,072023_Carolina_2023_Baya_055_MCAV_S4,072023_Carolina_2023_Baya_057_SSID_S7,072023_Carolina_2023_Baya_059_SSID_S8,072023_Carolina_2023_Baya_061_SSID_S5,072023_Carolina_2023_Baya_063_SSID_S6,072023_Carolina_2023_Baya_065_DCYL_S1,072023_Carolina_2023_Baya_066_DCYL_S2,122021_Coralina_2021_Baya_001_DCYL_S24,...,122021_Coralina_2021_Baya_021_SSID2_S36,122021_Coralina_2021_Baya_023_MCAV_S31,122021_Coralina_2021_Baya_025_MMEA_S22,122021_Coralina_2021_Baya_027_PSTR4_S34,122021_Coralina_2021_Baya_029_MMEA_S23,122021_Coralina_2021_Baya_031_MMEA_S45,122021_Coralina_2021_Baya_033_PSTR7_S35,122021_Coralina_2021_Baya_035_PSTR_S32,122021_Coralina_2021_Baya_037_PSTR_S33,Sum
3379134,4.956237e+05,501943.872023,653726.646612,560194.266796,543713.602329,579623.969058,618825.758308,540152.165079,621984.803033,588636.790302,...,497445.980163,485810.975674,558066.338471,501242.616992,570016.633320,598601.496562,501260.288066,533645.501150,528627.465257,1.534309e+07
1783272,3.431173e+05,336640.394298,250399.118640,334366.408762,340999.376040,353457.933503,347362.112630,308337.472344,294902.438894,331683.964921,...,354355.006318,356534.377221,317031.178544,355202.180827,306077.188246,292028.063458,351594.650206,328677.727095,336485.342701,9.149334e+06
Total_reads,1.170856e+06,141213.000000,303168.000000,137131.000000,192320.000000,394542.000000,827632.000000,88588.000000,187011.000000,74574.000000,...,409942.000000,94263.000000,74346.000000,440200.000000,41483.000000,85663.000000,155520.000000,148638.000000,109877.000000,6.409794e+06
4751,1.151038e+05,117446.694001,67272.931180,68853.869657,77854.617304,44157.529490,18201.326193,102519.528604,56440.530236,52002.038244,...,103687.838767,107444.066070,81941.193877,100624.716038,86589.687342,72936.973956,103729.423868,97713.908960,95397.580931,2.463905e+06
3366610,1.471488e+04,13759.356433,8919.147140,13680.349447,14392.678869,9773.357463,8072.428326,12067.097124,8486.131832,9708.477485,...,14924.062428,13579.028887,11675.140559,17578.373467,11016.561001,10389.549747,15123.456790,13993.729733,14598.141558,3.424558e+05
3384189,1.093217e+04,8936.854256,6029.660122,5396.299888,6026.414309,3259.475544,1840.189843,8274.258365,4432.894322,4036.259286,...,10145.337633,16846.482713,8527.694832,6288.050886,8099.703493,8988.711579,8551.954733,8127.127652,7590.305523,2.168060e+05
1783275,4.518062e+03,5580.222784,3615.157273,6592.236620,4944.883527,2303.937223,1027.026505,9583.690793,4325.948741,4022.849787,...,6076.469354,6068.128534,9469.238426,5477.055884,5881.927537,7844.693742,6063.528807,6223.173078,5915.705744,1.471097e+05
3384194,4.880190e+03,5204.903231,3562.381254,3587.810196,3899.750416,3140.350077,2733.098769,5858.581298,3823.304511,4103.306783,...,4315.244596,4147.968980,3550.964410,4447.978192,4218.595569,3513.769072,4308.127572,3989.558525,3922.567962,1.169489e+05
2731360,5.413988e+03,3795.684533,2477.174372,3945.132756,4279.326123,2364.767249,1267.471533,6050.480878,2711.070472,3003.727841,...,4256.699728,4010.056968,4492.508003,4173.103135,3664.151580,2428.119491,5369.084362,3727.176092,3840.658191,1.052281e+05
2732005,2.651052e+03,3590.321004,2163.816762,1735.566721,2007.071547,1206.462176,404.769269,6129.498352,2422.317404,1944.377397,...,2346.673432,2938.586720,4331.100530,3602.907769,2989.176289,2089.583601,2282.664609,2334.530874,1929.430181,7.205754e+04


In [35]:
# export otu table to csv
normalized_otu.to_csv("otu_normtable_kingdom.csv")